# Phase 10: Feature-Augmented Graph Neural Networks

In Phase 8, we built a **LightGCN** model that was incredibly effective at capturing collaborative filtering signals from the graph structure. However, it was "blind" to the content of the movies and the demographics of the users.

In this notebook, we implement a **Feature-Augmented GNN**. We take the raw features we engineered in Phase 5 and use them to initialize the GNN embeddings. This allows the model to make smarter predictions, especially for nodes with fewer connections (mitigating the Cold Start problem).

## 1. The Strategy
1. **Load Features:** Import User demographics and Movie genres.
2. **Linear Projection:** Project heterogeneous features (24D for users, 19D for movies) into a unified 64D embedding space.
3. **Graph Convolution:** Pass these feature-rich embeddings through LightGCN layers.
4. **Hybrid Learning:** Optimize with BPR Loss to learn the relationship between features and actual interactions.

In [2]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import pandas as pd
import numpy as np
from torch_geometric.nn import LGConv
from sklearn.preprocessing import MinMaxScaler, OneHotEncoder
from sklearn.model_selection import train_test_split
from torch.optim import Adam
import random

# Set seeds for reproducibility
torch.manual_seed(42)
np.random.seed(42)
random.seed(42)

c:\Users\RoG\AppData\Local\Programs\Python\Python313\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Step 1: Feature Loading and Processing

We replicate the logic from `feature_engineering.ipynb` to prepare our raw node features.

In [3]:
# --- 1. Load User Features ---
users_cols = ['user_id', 'age', 'gender', 'occupation', 'zip_code']
users = pd.read_csv('data/u.user', sep='|', names=users_cols)

scaler = MinMaxScaler()
users['age_norm'] = scaler.fit_transform(users[['age']])

encoder = OneHotEncoder(sparse_output=False)
encoded_features = encoder.fit_transform(users[['gender', 'occupation']])
encoded_df = pd.DataFrame(encoded_features, columns=encoder.get_feature_names_out(['gender', 'occupation']))
user_features_df = pd.concat([users[['user_id', 'age_norm']], encoded_df], axis=1).set_index('user_id')

# --- 2. Load Movie Features ---
genre_cols = [
    "unknown", "Action", "Adventure", "Animation", "Children's", "Comedy", 
    "Crime", "Documentary", "Drama", "Fantasy", "Film-Noir", "Horror", 
    "Musical", "Mystery", "Romance", "Sci-Fi", "Thriller", "War", "Western"
]
items_cols = ['movie_id', 'title', 'release_date', 'video_release_date', 'imdb_url'] + genre_cols
items = pd.read_csv('data/u.item', sep='|', names=items_cols, encoding='latin-1')
movie_features_df = items[['movie_id'] + genre_cols].set_index('movie_id')

print(f"User Features: {user_features_df.shape}")
print(f"Movie Features: {movie_features_df.shape}")

User Features: (943, 24)
Movie Features: (1682, 19)


## Step 2: Graph Data Setup

We map IDs to indices and create the `edge_index` for PyTorch Geometric.

In [4]:
df_ratings = pd.read_csv('data/u.data', sep='\t', names=['user_id', 'movie_id', 'rating', 'ts'])
pos_df = df_ratings[df_ratings['rating'] >= 4].copy()

user_map = {old: i for i, old in enumerate(sorted(df_ratings['user_id'].unique()))}
movie_map = {old: i for i, old in enumerate(sorted(df_ratings['movie_id'].unique()))}

pos_df['u_idx'] = pos_df['user_id'].map(user_map)
pos_df['m_idx'] = pos_df['movie_id'].map(movie_map)

train_edges, test_edges = train_test_split(pos_df, test_size=0.2, random_state=42)

edge_index = torch.stack([
    torch.tensor(train_edges['u_idx'].values, dtype=torch.long),
    torch.tensor(train_edges['m_idx'].values, dtype=torch.long)
])

# Convert features to Tensors (ordered by our mapping index)
user_feat_tensor = torch.tensor(user_features_df.loc[sorted(user_map.keys())].values, dtype=torch.float)
movie_feat_tensor = torch.tensor(movie_features_df.loc[sorted(movie_map.keys())].values, dtype=torch.float)

num_users, user_feat_dim = user_feat_tensor.shape
num_movies, movie_feat_dim = movie_feat_tensor.shape

## Step 3: Feature-Augmented LightGCN Model

This model differs from standard LightGCN by adding `linear_user` and `linear_movie` projection layers. Instead of learning embeddings from scratch, it learns to transform the raw features into the latent space.

In [5]:
class FeatureAugmentedLightGCN(nn.Module):
    def __init__(self, num_users, num_items, user_feat_dim, item_feat_dim, embedding_dim=64, num_layers=3):
        super(FeatureAugmentedLightGCN, self).__init__()
        self.num_users = num_users
        self.num_items = num_items
        self.embedding_dim = embedding_dim
        self.num_layers = num_layers

        # 1. Feature Projection Layers
        # These map raw features into the 64D embedding space
        self.linear_user = nn.Linear(user_feat_dim, embedding_dim)
        self.linear_item = nn.Linear(item_feat_dim, embedding_dim)

        # 2. Collaborative Filtering Embeddings (Residual)
        # We still keep a learnable embedding to capture signals NOT in the features
        self.user_emb_res = nn.Embedding(num_users, embedding_dim)
        self.item_emb_res = nn.Embedding(num_items, embedding_dim)
        
        nn.init.normal_(self.user_emb_res.weight, std=0.1)
        nn.init.normal_(self.item_emb_res.weight, std=0.1)

        # 3. LightGCN Convolution layers
        self.convs = nn.ModuleList([LGConv() for _ in range(num_layers)])

    def forward(self, edge_index, user_features, item_features):
        # Map features to embedding space
        u_feat_emb = self.linear_user(user_features)
        i_feat_emb = self.linear_item(item_features)
        
        # Combine features + residual learned embeddings
        # This is a 'Hybrid Initialization'
        u_emb = u_feat_emb + self.user_emb_res.weight
        i_emb = i_feat_emb + self.item_emb_res.weight
        
        x = torch.cat([u_emb, i_emb], dim=0)
        
        # Bipartite Edge Index Adjustment
        user_indices, item_indices = edge_index[0], edge_index[1]
        adj_edge_index = torch.stack([
            torch.cat([user_indices, item_indices + self.num_users]),
            torch.cat([item_indices + self.num_users, user_indices])
        ], dim=0)

        all_layers = [x]
        for conv in self.convs:
            x = conv(x, adj_edge_index)
            all_layers.append(x)
        
        final_embeddings = torch.mean(torch.stack(all_layers, dim=0), dim=0)
        users_emb, items_emb = torch.split(final_embeddings, [self.num_users, self.num_items])
        return users_emb, items_emb

model = FeatureAugmentedLightGCN(num_users, num_movies, user_feat_dim, movie_feat_dim)
print(model)

FeatureAugmentedLightGCN(
  (linear_user): Linear(in_features=24, out_features=64, bias=True)
  (linear_item): Linear(in_features=19, out_features=64, bias=True)
  (user_emb_res): Embedding(943, 64)
  (item_emb_res): Embedding(1682, 64)
  (convs): ModuleList(
    (0-2): 3 x LGConv()
  )
)


## Step 4: Training with BPR Loss

We use the same BPR loss and negative sampling as in Phase 8 to ensure our comparison is valid.

In [6]:
def get_bpr_loss(users_emb, pos_items_emb, neg_items_emb, lambda_reg=1e-4):
    pos_scores = torch.mul(users_emb, pos_items_emb).sum(dim=1)
    neg_scores = torch.mul(users_emb, neg_items_emb).sum(dim=1)
    loss = -torch.log(torch.sigmoid(pos_scores - neg_scores)).mean()
    reg_loss = lambda_reg * (users_emb.norm(2).pow(2) + pos_items_emb.norm(2).pow(2) + neg_items_emb.norm(2).pow(2)) / users_emb.size(0)
    return loss + reg_loss

def sample_negative_items(edge_index, num_items):
    users, pos_items = edge_index[0], edge_index[1]
    neg_items = []
    for u in users:
        while True:
            neg_i = random.randint(0, num_items - 1)
            if neg_i not in pos_items[users == u]:
                neg_items.append(neg_i)
                break
    return torch.tensor(neg_items, dtype=torch.long)

# --- Start Training ---
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = model.to(device)
edge_index = edge_index.to(device)
u_feat_tensor = user_feat_tensor.to(device)
m_feat_tensor = movie_feat_tensor.to(device)
optimizer = Adam(model.parameters(), lr=0.01)

print(f"Training Augmented GNN on {device}...")
for epoch in range(1, 101):
    model.train()
    optimizer.zero_grad()
    
    u_emb, i_emb = model(edge_index, u_feat_tensor, m_feat_tensor)
    neg_indices = sample_negative_items(edge_index, num_movies).to(device)
    
    loss = get_bpr_loss(u_emb[edge_index[0]], i_emb[edge_index[1]], i_emb[neg_indices])
    
    loss.backward()
    optimizer.step()
    
    if epoch % 10 == 0:
        print(f"Epoch {epoch:03d} | Loss: {loss.item():.4f}")

Training Augmented GNN on cpu...
Epoch 010 | Loss: 0.3414
Epoch 020 | Loss: 0.2924
Epoch 030 | Loss: 0.2516
Epoch 040 | Loss: 0.2204
Epoch 050 | Loss: 0.2003
Epoch 060 | Loss: 0.1894
Epoch 070 | Loss: 0.1786
Epoch 080 | Loss: 0.1675
Epoch 090 | Loss: 0.1638
Epoch 100 | Loss: 0.1536


## Step 5: Evaluation and Benchmarking

We check the Precision@10 to see if feature injection improved our score.

In [7]:
def recommend_augmented(u_node, model, edge_index, user_feat_tensor, movie_feat_tensor, k=10):
    model.eval()
    with torch.no_grad():
        u_emb, i_emb = model(edge_index, user_feat_tensor, movie_feat_tensor)
        u_idx = user_map[int(u_node.replace('u_', ''))]
        
        scores = torch.matmul(u_emb[u_idx], i_emb.t())
        seen = edge_index[1][edge_index[0] == u_idx].tolist()
        scores[seen] = -np.inf
        
        top_k = torch.topk(scores, k).indices.cpu().numpy()
        rev_movie_map = {v: k for k, v in movie_map.items()}
        return ['m_' + str(rev_movie_map[idx]) for idx in top_k]

precisions, recalls, mrrs = [], [], []
test_users = test_edges['u_idx'].unique()[:100]
for u_idx in test_users:
    u_id = list(user_map.keys())[list(user_map.values()).index(u_idx)]
    hidden = set('m_' + str(list(movie_map.keys())[list(movie_map.values()).index(m)]) 
                 for m in test_edges[test_edges['u_idx'] == u_idx]['m_idx'])
    
    recs = recommend_augmented(f'u_{u_id}', model, edge_index, u_feat_tensor, m_feat_tensor)
    hits = set(recs).intersection(hidden)
    
    # Precision@10
    precisions.append(len(hits) / len(recs))
    
    # Recall@10
    recalls.append(len(hits) / len(hidden) if hidden else 0)
    
    # MRR (Mean Reciprocal Rank)
    mrr = 0
    for i, rec in enumerate(recs):
        if rec in hidden:
            mrr = 1 / (i + 1)
            break
    mrrs.append(mrr)

print(f"\nAugmented GNN Results:")
print(f"- Mean Precision@10: {np.mean(precisions):.4f}")
print(f"- Mean Recall@10:    {np.mean(recalls):.4f}")
print(f"- Mean MRR:          {np.mean(mrrs):.4f}")


Augmented GNN Results:
- Mean Precision@10: 0.3390
- Mean Recall@10:    0.1697
- Mean MRR:          0.6085


## Final Conclusion & Comparison

By augmenting LightGCN with demographic and genre features, we have achieved the highest performance in the project to date.

### The Leaderboard
| Model | Precision@10 | Recall@10 | MRR |
| :--- | :--- | :--- | :--- |
| Node2Vec/Metapath2Vec | < 0.010 | < 0.010 | < 0.010 |
| Hybrid Model | 0.103 | 0.089 | 0.275 |
| Jaccard Baseline | 0.146 | 0.168 | 0.381 |
| LightGCN (Pure) | 0.144 | 0.179 | 0.375 |
| **Augmented GNN (Phase 10)** | **0.3390** | **0.1697** | **0.6085** |

### Why did we win by such a large margin?
- **Ranking Precision & MRR:** The jump in MRR from **0.375 to 0.608** is the most significant insight. It means that when the model is correct, it usually ranks the movie in the **1st or 2nd spot**, whereas Jaccard and Pure LightGCN averaged the 3rd or 4th spot. The model is now much more *certain* about its top picks.
- **Feature Propagation:** Standard LightGCN only propagates "node IDs". In our Augmented model, we propagate the *intent*. The model understands that 'Students' who like 'Action' movies share a latent preference, even if they haven't seen the exact same movies yet.
- **Warm Start:** By using Linear Projection layers, we avoid the 'Cold Start' problem. Every node starts in a meaningful position in the latent space rather than a random one.

### Next Steps
The current model treats all positive edges as equal (rating 4 or 5). The next logical evolution is **Weighted Link Prediction**, where we incorporate the actual rating values into the message-passing layers to distinguish between 'Liking' and 'Loving' a movie.